In [1]:
import subprocess
subprocess.run(["pip", "install", "boto3"], check=True)

CompletedProcess(args=['pip', 'install', 'boto3'], returncode=0)

In [5]:
spark.stop()

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MinIO-Finance") \
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.367,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "s3a://finance/iceberg/") \
    .getOrCreate()

print(spark.version)

3.5.0


In [3]:
# Create a Java URI for your MinIO bucket
uri = spark._jvm.java.net.URI("s3a://finance/")

# Pass both the URI and the Hadoop configuration to resolve the correct filesystem
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(uri, spark._jsc.hadoopConfiguration())

# Now list the files
files = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path("s3a://finance/"))

for f in files:
    print(f.getPath())

s3a://finance/iceberg
s3a://finance/parquet
s3a://finance/processed
s3a://finance/raw


In [4]:
from pyspark.sql.functions import input_file_name, regexp_extract

df_parsed = spark.read \
    .option("delimiter", "\t") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3a://finance/processed/spacy_pipeline/parsed_txt/") \
    .withColumn("source_file", input_file_name()) \
    .withColumn("company", regexp_extract(input_file_name(), r"/([A-Z]+)_[A-Z]+_(\d{4})", 1)) \
    .withColumn("year", regexp_extract(input_file_name(), r"/[A-Z]+_[A-Z]+_(\d{4})", 1).cast("integer"))

df_parsed.printSchema()
df_parsed.show(5)

root
 |-- TOKEN: string (nullable = true)
 |-- LEMMA: string (nullable = true)
 |-- POS: string (nullable = true)
 |-- TAG: string (nullable = true)
 |-- DEP: string (nullable = true)
 |-- HEAD: string (nullable = true)
 |-- source_file: string (nullable = false)
 |-- company: string (nullable = false)
 |-- year: integer (nullable = true)

+----------+----------+-----+---+--------+----------+--------------------+-------+----+
|     TOKEN|     LEMMA|  POS|TAG|     DEP|      HEAD|         source_file|company|year|
+----------+----------+-----+---+--------+----------+--------------------+-------+----+
|    UNITED|    UNITED|PROPN|NNP|compound|COMMISSION|s3a://finance/pro...|   AAPL|2017|
|    STATES|    STATES|PROPN|NNP|    nmod|COMMISSION|s3a://finance/pro...|   AAPL|2017|
|SECURITIES|SECURITIES|PROPN|NNP|    nmod|COMMISSION|s3a://finance/pro...|   AAPL|2017|
|       AND|       and|CCONJ| CC|      cc|SECURITIES|s3a://finance/pro...|   AAPL|2017|
|  EXCHANGE|  EXCHANGE|PROPN|NNP|    conj|

In [5]:
# Save parsed data as Parquet
df_parsed.write.mode("overwrite") \
    .partitionBy("company") \
    .parquet("s3a://finance/parquet/spacy_pipeline/parsed_txt/")

print("Done!")

Done!


In [6]:
# Read back to verify
df_verify = spark.read.parquet("s3a://finance/parquet/spacy_pipeline/parsed_txt/")
df_verify.printSchema()
df_verify.count()

root
 |-- TOKEN: string (nullable = true)
 |-- LEMMA: string (nullable = true)
 |-- POS: string (nullable = true)
 |-- TAG: string (nullable = true)
 |-- DEP: string (nullable = true)
 |-- HEAD: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- company: string (nullable = true)



1055474

In [7]:
# Read your existing Parquet
df_parsed = spark.read.parquet("s3a://finance/parquet/spacy_pipeline/parsed_txt/")

# Write to Iceberg table
df_parsed.writeTo("local.spacy_parsed") \
    .partitionedBy("company") \
    .createOrReplace()

print("Iceberg table created!")

Iceberg table created!


In [8]:
spark.sql("SELECT company, COUNT(*) as count FROM local.spacy_parsed GROUP BY company ORDER BY company").show()

+-------+------+
|company| count|
+-------+------+
|   AAPL|667363|
|   AMZN|388111|
+-------+------+



In [9]:
# DATA INTEGRITY CHECK
# 1. Check total row count
print("Total rows:", df_parsed.count())

# 2. Check for null values in each column
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_parsed.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_parsed.columns
])
print("Null counts per column:")
null_counts.show()

# HEAD column contains null values, expected behavior as ROOT tokens in dependency parsing have no head word.

# 3. Check distinct companies
print("Distinct companies:")
df_parsed.select("company").distinct().show()

# 4. Check schema is intact
df_parsed.printSchema()

Total rows: 1055474
Null counts per column:
+-----+-----+---+---+---+----+-----------+----+-------+
|TOKEN|LEMMA|POS|TAG|DEP|HEAD|source_file|year|company|
+-----+-----+---+---+---+----+-----------+----+-------+
|    0|    0|  0|  0|  0|6356|          0|   0|      0|
+-----+-----+---+---+---+----+-----------+----+-------+

Distinct companies:
+-------+
|company|
+-------+
|   AAPL|
|   AMZN|
+-------+

root
 |-- TOKEN: string (nullable = true)
 |-- LEMMA: string (nullable = true)
 |-- POS: string (nullable = true)
 |-- TAG: string (nullable = true)
 |-- DEP: string (nullable = true)
 |-- HEAD: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- company: string (nullable = true)



In [10]:
# READING DATA FROM PARQUET AND ICEBERG
# 1. Read back from Parquet
print("=== Reading from Parquet ===")
df_parquet = spark.read.parquet("s3a://finance/parquet/spacy_pipeline/parsed_txt/")
print("Row count:", df_parquet.count())
df_parquet.show(5)

# 2. Read back from Iceberg
print("=== Reading from Iceberg ===")
df_iceberg = spark.sql("SELECT * FROM local.spacy_parsed LIMIT 5")
df_iceberg.show(5)

# 3. Example analysis query - filter by company
print("=== Sample Analysis: AAPL nouns only ===")
spark.sql("""
    SELECT TOKEN, LEMMA, POS, DEP 
    FROM local.spacy_parsed 
    WHERE company = 'AAPL' AND POS = 'NOUN'
    LIMIT 10
""").show()

=== Reading from Parquet ===
Row count: 1055474
+----------+----------+-----+---+--------+----------+--------------------+----+-------+
|     TOKEN|     LEMMA|  POS|TAG|     DEP|      HEAD|         source_file|year|company|
+----------+----------+-----+---+--------+----------+--------------------+----+-------+
|    UNITED|    UNITED|PROPN|NNP|compound|COMMISSION|s3a://finance/pro...|2017|   AAPL|
|    STATES|    STATES|PROPN|NNP|    nmod|COMMISSION|s3a://finance/pro...|2017|   AAPL|
|SECURITIES|SECURITIES|PROPN|NNP|    nmod|COMMISSION|s3a://finance/pro...|2017|   AAPL|
|       AND|       and|CCONJ| CC|      cc|SECURITIES|s3a://finance/pro...|2017|   AAPL|
|  EXCHANGE|  EXCHANGE|PROPN|NNP|    conj|SECURITIES|s3a://finance/pro...|2017|   AAPL|
+----------+----------+-----+---+--------+----------+--------------------+----+-------+
only showing top 5 rows

=== Reading from Iceberg ===
+----------+----------+-----+---+--------+----------+--------------------+----+-------+
|     TOKEN|     L